In [1]:
from selenium import webdriver
from bs4 import BeautifulSoup as bs
import pandas as pd
import requests 

In [2]:
# 무신사의 반팔 상의 주소를 저장 
url = 'https://www.musinsa.com/category/001001/goods?gf=A'

# 상품에 대한 정보(브랜명, 상품 이름, 상품의 할인율, 가격)

res = requests.get(url)
res

<Response [200]>

In [3]:
soup = bs(res.text, 'html.parser')
soup

<!DOCTYPE html>
<html lang="ko-KR"><head><meta charset="utf-8" data-next-head=""/><title data-next-head="">반소매 티셔츠 | 무신사 추천 상품</title><meta content="width=device-width, initial-scale=1" data-next-head="" name="viewport"/><meta content="무신사 추천 상품" data-next-head="" data-rh="true" name="description"/><meta content="website" data-next-head="" property="og:type"/><meta content="반소매 티셔츠 | 무신사 추천 상품" data-next-head="" data-rh="true" property="og:title"/><meta content="무신사 추천 상품" data-next-head="" data-rh="true" property="og:description"/><meta content="https://image.msscdn.net/static/assets/bi/og/og_musinsa.png" data-next-head="" name="og:image" property="og:image"/><link data-next-head="" href="https://www.musinsa.com/category/001001" rel="canonical"/><link data-next-head="" href="https://www.musinsa.com/category/001001" hreflang="ko-KR" rel="alternate"/><link data-next-head="" href="https://static.msscdn.net/static/v2/pc/category/vendor.js" rel="modulepreload"/><link data-next-head="" href

1. selenium을 이용하여 무신사 페이지에 요청
2. 해당 페이지의 html 문서를 불러온다
3. bs4를 이용하여 데이터 파싱
4. GoodsList__List로 시작하는 class 값을 가진 div 태그를 찾는다.
5. GoodsList__Row로 시작하는 class 값을 가진 모든 div 태그를 찾는다
6. sc-it로 시작하는 class 값을 가진 모든 div 태그를 찾는다.
7. 브랜드명, 이름, 할인율, 가격 데이터, 해당 상품의 링크 주소를 추출
8. 추출한 데이터를 데이터프레임으로 생성

In [4]:
import re
import time

In [5]:
driver = webdriver.Chrome()

In [6]:
# 무신사 페이지에 요청
driver.get(url)

In [7]:
time.sleep(5)   # ← 이 줄 추가

In [8]:
soup = bs(driver.page_source, 'html.parser')

In [9]:
soup

<html lang="ko-KR"><head><link href="https://static.msscdn.net/static/mds/2.0.0/mds-with-prefix.css" rel="stylesheet" type="text/css"/><meta charset="utf-8" data-next-head=""/><title data-next-head="">반소매 티셔츠 | 무신사 추천 상품</title><meta content="width=device-width, initial-scale=1" data-next-head="" name="viewport"/><meta content="website" data-next-head="" property="og:type"/><meta content="https://image.msscdn.net/static/assets/bi/og/og_musinsa.png" data-next-head="" name="og:image" property="og:image"/><link data-next-head="" href="https://www.musinsa.com/category/001001" rel="canonical"/><link data-next-head="" href="https://www.musinsa.com/category/001001" hreflang="ko-KR" rel="alternate"/><link data-next-head="" href="https://static.msscdn.net/static/v2/pc/category/vendor.js" rel="modulepreload"/><link data-next-head="" href="https://static.msscdn.net/static/v2/pc/category/category.js" rel="modulepreload"/><link href="https://static.msscdn.net/static/common/layout-pc/style.css" rel=

In [10]:
# class이 값이 특정 문자로 시작하는? 
# re.compile(r"^GoddsList__List")
div_tag = soup.find('div', class_=re.compile(r"^GoodsList__List")) 

In [11]:
goods_row = div_tag.find_all('div', class_= re.compile(r"^GoodsList__Row"))


In [12]:
# goods_row에서 각각의 원소에서 sc-it로 시작하는 class 값을 가진 div 태그들을 모두 찾는다. 
goods_dict = []
for good_info in goods_row:
    info_data = good_info.find_all('div', class_=re.compile(r'^sc-it'))
    for data in info_data:
        # 상품의 정보를 저장할수 있는 딕셔너리형 데이터 초기값을 저장 
        info_dict = {}
        # info_dict에서 사용할 키 값들의 목록 생성 
        dict_keys = ['브랜드', '상품명', '할인율', '판매가격']
        # print( len(data.find_all('span')) )
        span_tags = data.find_all('span')
        for span, k in zip(span_tags, dict_keys):
            text = span.get_text()
            # info_dict에 데이터를 추가 
            info_dict[k] = text.strip()
            # print(info_dict)
        # 해당 상품의 하이퍼링크 주소 값을 info_dict에 추가 
        # data에서 a태그들을 찾아서 2번째 a 태그의 href 값을 추출
        link_url = data.find_all('a')[1]['href']
        info_dict['link'] = link_url
        # break
        # 3번째 반복문이 끝나고 만들어진 상품 정보 데이터를 goods_dict에 추가 
        goods_dict.append(info_dict)
    # break

In [13]:
goods_dict

[{'브랜드': '소버먼트',
  '상품명': '【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color]',
  '할인율': '30%',
  '판매가격': '29,000원',
  'link': 'https://www.musinsa.com/products/6659876'},
 {'브랜드': '누아트 스튜디오',
  '상품명': '[송승일 PICK] 우먼 솔리드 반팔 티셔츠_4colors',
  '할인율': '41%',
  '판매가격': '18,500원',
  'link': 'https://www.musinsa.com/products/6433549'},
 {'브랜드': '이스케이프프롬',
  '상품명': '하트 포인트 ESCF 프린트 보트넥 슬림핏 반팔티 [4color]',
  '할인율': '29%',
  '판매가격': '36,900원',
  'link': 'https://www.musinsa.com/products/6612061'},
 {'브랜드': '엘엠알',
  '상품명': '[지현서 PICK] 도트 셔링 반팔 티셔츠 DARK GREY',
  '할인율': '20%',
  '판매가격': '39,100원',
  'link': 'https://www.musinsa.com/products/6453175'},
 {'브랜드': '그로우하이드',
  '상품명': '포헬 원오프 오버핏 레터링 반팔 티셔츠_브라운',
  '할인율': '69,000원',
  'link': 'https://www.musinsa.com/products/6581346'},
 {'브랜드': '페이드',
  '상품명': 'PD ase 슬림핏 반팔티 블랙',
  '할인율': '34%',
  '판매가격': '24,900원',
  'link': 'https://www.musinsa.com/products/6548127'},
 {'브랜드': '프리즘웍스',
  '상품명': 'ZANES BAIT & TACKLE RINGER TEE _ IVORY',
  '할인율': '10%',
  '판매가격': 

In [14]:
# driver에서 화면 스크롤 이벤트
driver.execute_script(
    'window.scrollBy(0, 1600);'
)

In [15]:
df = pd.DataFrame(goods_dict)
df

,브랜드,상품명,할인율,판매가격,link
0,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],30%,"29,000원",https://www.musinsa.com/products/6659876
1,누아트 스튜디오,[송승일 PICK] 우먼 솔리드 반팔 티셔츠_4colors,41%,"18,500원",https://www.musinsa.com/products/6433549
2,이스케이프프롬,하트 포인트 ESCF 프린트 보트넥 슬림핏 반팔티 [4color],29%,"36,900원",https://www.musinsa.com/products/6612061
3,엘엠알,[지현서 PICK] 도트 셔링 반팔 티셔츠 DARK GREY,20%,"39,100원",https://www.musinsa.com/products/6453175
4,그로우하이드,포헬 원오프 오버핏 레터링 반팔 티셔츠_브라운,"69,000원",NaN,https://www.musinsa.com/products/6581346
5,페이드,PD ase 슬림핏 반팔티 블랙,34%,"24,900원",https://www.musinsa.com/products/6548127
6,프리즘웍스,ZANES BAIT & TACKLE RINGER TEE _ IVORY,10%,"40,500원",https://www.musinsa.com/products/6490668
7,일꼬르소,CRS Sorona Cotton ARCHIVE 숏 슬리브 티셔츠 블랙,5%,"56,050원",https://www.musinsa.com/products/6581788
8,해비즘,NAME CARD S/S TEE(PLUM),10%,"37,800원",https://www.musinsa.com/products/6579994
9,엘엠알,[지현서 PICK] 투웨이 레이스 셔링 레이어드 반팔 티셔츠 IVORY / SKY ...,43%,"39,100원",https://www.musinsa.com/products/6453251


In [16]:
print(df.columns)

Index(['브랜드', '상품명', '할인율', '판매가격', 'link'], dtype='str')


In [17]:
print(len(goods_row))  # 0이면 파싱 실패
print(df)

9
             브랜드                                                상품명      할인율  \
0           소버먼트           【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color]      30%   
1       누아트 스튜디오                   [송승일 PICK] 우먼 솔리드 반팔 티셔츠_4colors      41%   
2        이스케이프프롬               하트 포인트 ESCF 프린트 보트넥 슬림핏 반팔티 [4color]      29%   
3            엘엠알                  [지현서 PICK] 도트 셔링 반팔 티셔츠 DARK GREY      20%   
4         그로우하이드                          포헬 원오프 오버핏 레터링 반팔 티셔츠_브라운  69,000원   
5            페이드                                  PD ase 슬림핏 반팔티 블랙      34%   
6          프리즘웍스             ZANES BAIT & TACKLE RINGER TEE _ IVORY      10%   
7           일꼬르소             CRS Sorona Cotton ARCHIVE 숏 슬리브 티셔츠 블랙       5%   
8            해비즘                            NAME CARD S/S TEE(PLUM)      10%   
9            엘엠알  [지현서 PICK] 투웨이 레이스 셔링 레이어드 반팔 티셔츠 IVORY / SKY ...      43%   
10           해칭룸                                    Lines Tee Green  68,000원   
11           무센트                   사하라

In [18]:
link_list = df['link'].tolist()
name_list = df['상품명'].tolist()
link_list[0]

'https://www.musinsa.com/products/6659876'

In [26]:
driver = webdriver.Chrome()

In [27]:
driver.get(link_list[0])

In [28]:
time.sleep(3)   # ← 추가 (페이지 로딩 대기)

In [29]:
# 스크롤을 마지막까지 내린다.
driver.execute_script(
    'window.scrollBy(0, document.body.scrollHeight);'
)

In [30]:
soup2 = bs(driver.page_source,'html.parser')

In [31]:
# GoodReviewListSection로 시작하는 class 값을 가진 div 태그를 선택
reviews_tag = soup2.find('div', class_=re.compile(f'GoodsReviewListSection'))
reviews_tag

<div class="GoodsReviewListSection__Container-sc-hmqz8q-0"><div class="GoodsReviewFilterTabSection__ScrollAnchor-sc-yqbv7s-0"></div><div class="GoodsReviewTabGroup__Container-sc-1mdjrqg-0 ffSVcu"><div class="flex w-full scrollbar-hide overflow-x-scroll GoodsReviewTabGroup__TabGroup-sc-1mdjrqg-1 zLJqW" data-mds="TabFixed"><div class="GoodsReviewTabGroup__TabItemWrapper-sc-1mdjrqg-3 CiRGj gtm-click-button" data-applied-tab="(not set)" data-brand-id="(not set)" data-button-id="전체" data-button-name="전체" data-index="(not set)" data-section-index="42" data-section-name="review_tab" data-selected="true" data-testid="goods-review-tab-전체"><span class="px-4 py-2 cursor-pointer bg-white text-black text-[14px] leading-[20px] tracking-[0] font-[600] lang-ja-JP:text-[13px] lang-zh-CN:text-[13px] lang-zh-TW:text-[13px] GoodsReviewTabGroup__TabItem-sc-1mdjrqg-2 hNLXRf" data-mds="TabFixedItem"><span class="relative" data-mds="TabFixedItem"><span class="text-[13px] leading-[18px] tracking-[0] font-[600]

In [32]:
# ExpandableContent로 시작하는 class 값을 가진 div 태그를 모두 찾는다
review_tags = reviews_tag.find_all('div', class_ = re.compile(r'^ExpandableContent'))
review_tags

[<div class="ExpandableContent__Container-sc-8yrutl-0 iuidVu gtm-click-button" data-applied-tab="(not set)" data-brand-id="(not set)" data-button-id="review_content" data-button-name="후기내용" data-index="(not set)" data-section-index="31" data-section-name="review"><div class="ExpandableContent__ContentContainer-sc-8yrutl-1 gvkPRD"><div style="max-height: 63px; overflow: hidden; white-space: pre-line;"></div><div style="white-space: pre-line;"><span class="text-[13px] leading-[18px] tracking-[0] font-[400] lang-ja-JP:text-[12px] lang-zh-CN:text-[12px] lang-zh-TW:text-[12px] w-full text-black font-global" data-mds="Typography">여름에 입기 좋은 두께감에 소재도 시원해서 너무 좋아용</span></div><div aria-hidden="true" class="Truncate__MeasureContainer-sc-1gm444l-0 ljgstB" style="max-width: 664.656px;"><span class="text-[13px] leading-[18px] tracking-[0] font-[400] lang-ja-JP:text-[12px] lang-zh-CN:text-[12px] lang-zh-TW:text-[12px] w-full text-black font-global" data-mds="Typography">여름에 입기 좋은 두께감에 소재도 시원해서 너무 좋아용

In [33]:
[tag.get_text().replace('\n','') for tag in review_tags]

['여름에 입기 좋은 두께감에 소재도 시원해서 너무 좋아용여름에 입기 좋은 두께감에 소재도 시원해서 너무 좋아용\xa0...더보기',
 '여름에 입기 좋은 두께감에 소재도 시원해서 너무 좋아용여름에 입기 좋은 두께감에 소재도 시원해서 너무 좋아용\xa0...더보기',
 '예상 예약 배송일정보다 늦어서 좀 그랬는데생각보다 만족스러워요각잡힌 빳빳한 원단이 아니라 보들거리는 원단이라\xa0...더보기예상 예약 배송일정보다 늦어서 좀 그랬는데생각보다 만족스러워요각잡힌 빳빳한 원단이 아니라 보들거리는 원단이라\xa0...더보기',
 '예상 예약 배송일정보다 늦어서 좀 그랬는데생각보다 만족스러워요각잡힌 빳빳한 원단이 아니라 보들거리는 원단이라\xa0...더보기예상 예약 배송일정보다 늦어서 좀 그랬는데생각보다 만족스러워요각잡힌 빳빳한 원단이 아니라 보들거리는 원단이라\xa0...더보기',
 '피그먼트 반팔이랑 다르게 재질이 부드러운 편이지만 한여름에도 편하게 잘 입을것 같습니당 다른 컬러도 구매하고 싶네요피그먼트 반팔이랑 다르게 재질이 부드러운 편이지만 한여름에도 편하게 잘 입을것 같습니당 다른 컬러도 구매하고 싶네요\xa0...더보기',
 '피그먼트 반팔이랑 다르게 재질이 부드러운 편이지만 한여름에도 편하게 잘 입을것 같습니당 다른 컬러도 구매하고 싶네요피그먼트 반팔이랑 다르게 재질이 부드러운 편이지만 한여름에도 편하게 잘 입을것 같습니당 다른 컬러도 구매하고 싶네요\xa0...더보기',
 '피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서한여름에도 편하게 잘 입을것 같습니당다른 컬러도 구매하고 싶어용!!!피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서한여름에도 편하게 잘 입을것 같습니당다른 컬러도 구매하고 싶어용!!!\xa0...더보기',
 '피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서한여름에도 편하게 잘 입을것 같습니당다른 컬러도 구매하고 싶어용!!!피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서한여름에도 편하게 잘 입을것 같습니당다른 컬러도 구매하고 싶어용

In [34]:
import time

In [35]:
driver = webdriver.Chrome()

In [36]:
# 반복문 생성 (상품명 리스트와 link를 이용해서)
review_dict =[]
for name, link in zip(name_list, link_list):
    # print(link)
    # break
    driver.get(link)
    time.sleep(1)
    soup2 = bs(driver.page_source, 'html.parser')
    reviews_tag = soup2.find('div', class_=re.compile(f"^GoodsReviewListSection"))
    try:
        review_tags = reviews_tag.find_all('div', class_=re.compile(r"^ExpandableContent"))
        for tag in review_tags:
            text = tag.get_text().replace('\n', '')
            review_dict.append(
                {
                    '상품명' : name, 
                    '리뷰' : text
                }
            )
    except:
        pass

In [37]:

driver.close()

In [38]:
review_df = pd.DataFrame(review_dict)

In [39]:
# 중복 리뷰는 제거 
review_df.drop_duplicates('리뷰', inplace=True)

In [40]:
# df 와 review_df를 결합(조인 결합)

total_df = pd.merge(df, review_df, on = '상품명', how='left') 

In [41]:
total_df

,브랜드,상품명,할인율,판매가격,link,리뷰
0,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],30%,"29,000원",https://www.musinsa.com/products/6659876,여름에 입기 좋은 두께감에 소재도 시원해서 너무 좋아용여름에 입기 좋은 두께감에 소...
1,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],30%,"29,000원",https://www.musinsa.com/products/6659876,예상 예약 배송일정보다 늦어서 좀 그랬는데생각보다 만족스러워요각잡힌 빳빳한 원단이 ...
2,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],30%,"29,000원",https://www.musinsa.com/products/6659876,피그먼트 반팔이랑 다르게 재질이 부드러운 편이지만 한여름에도 편하게 잘 입을것 같습...
3,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],30%,"29,000원",https://www.musinsa.com/products/6659876,피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서한여름에도 편하게 잘 입을것 같습니...
4,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],30%,"29,000원",https://www.musinsa.com/products/6659876,피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서 한여름에도 편하게 잘 입을것 같습...
...,...,...,...,...,...,...
162,오오엠엘,피플 슬림 숏 슬리브 멜란지 그레이,20%,"29,520원",https://www.musinsa.com/products/6508556,체험단 이 후기는 제품을 무상으로 제공받아 직접 사용해 본 후 작성되었습니다.배송받...
163,오오엠엘,피플 슬림 숏 슬리브 멜란지 그레이,20%,"29,520원",https://www.musinsa.com/products/6508556,이런 기본티 예쁜걸 찾고 있었는데 프린팅도 예쁘고 핏도 딱 제가 원하던 핏이라 만족...
164,오오엠엘,피플 슬림 숏 슬리브 멜란지 그레이,20%,"29,520원",https://www.musinsa.com/products/6508556,체험단 이 후기는 제품을 무상으로 제공받아 직접 사용해 본 후 작성되었습니다.평소 ...
165,오드스튜디오,"[디지몬 어드벤처] Vamdemon, Gottsumon, and Pumpmon Ov...",10%,"37,800원",https://www.musinsa.com/products/6546927,예뻐요! 프린팅도 진하게 나오고 같이온 스티커도 귀엽고 택에도 아구몬ㅠㅠ 넘 귀여워...


In [42]:
len(review_df['상품명'].unique())

28